# Mercor Cheating Detection — EDA & Feature Exploration
### Week 1: Data Understanding · Feature Selection · Baseline

**Cost table (leaderboard metric):**
| Decision | Error | Cost |
|---|---|---|
| Auto-pass a cheater | False Negative | **$600** |
| Auto-block a legit user | FP block | **$300** |
| Review a legit user | FP review | **$150** |
| Review a cheater | TP review | **$5** |
| Correct auto-pass / block | — | **$0** |

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import PATHS
from src.utils.cost_metric import competition_cost, find_best_thresholds

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Data Overview

In [ ]:
train = pd.read_csv(PATHS['train_file'])
test  = pd.read_csv(PATHS['test_file'])

labeled   = train[train['is_cheating'].notna()].copy()
unlabeled = train[train['is_cheating'].isna()].copy()

print(f'Train rows       : {len(train):,}')
print(f'Labeled          : {len(labeled):,}  ({len(labeled)/len(train):.1%})')
print(f'Cheaters         : {(labeled.is_cheating==1).sum():,}  ({(labeled.is_cheating==1).mean():.2%})')
print(f'Legit (labeled)  : {(labeled.is_cheating==0).sum():,}')
print(f'Unlabeled        : {len(unlabeled):,}')
print(f'high_conf_clean  : {(train.high_conf_clean==1).sum():,}')
print(f'Test rows        : {len(test):,}')

## 2. Missing Value Analysis

In [ ]:
feat_cols = [f'feature_{i:03d}' for i in range(1, 19)]
missing = labeled[feat_cols].isnull().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
missing.plot.bar(ax=ax, color='steelblue')
ax.set_title('Missing Value Rate by Feature (Labeled Set)')
ax.set_ylabel('Missing %')
ax.axhline(0.10, color='red', linestyle='--', label='10% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Distribution: Cheater vs. Legit

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(feat_cols):
    ax = axes[i]
    for label, grp in labeled.groupby('is_cheating'):
        grp[col].dropna().hist(ax=ax, bins=40, alpha=0.6,
                               label='Cheater' if label == 1 else 'Legit',
                               density=True)
    ax.set_title(col, fontsize=8)
    ax.set_xlabel('')
    if i == 0:
        ax.legend(fontsize=7)

# Hide unused axes
for j in range(len(feat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Cheater vs. Legit', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/plots/feature_distributions.png', bbox_inches='tight')
plt.show()

## 4. Feature 010 — Log Transform Justification

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

labeled['feature_010'].dropna().hist(bins=100, ax=ax1)
ax1.set_title('feature_010 — Raw (0–39M range)')

np.log1p(labeled['feature_010'].dropna()).hist(bins=100, ax=ax2, color='salmon')
ax2.set_title('feature_010 — log1p transformed')

plt.tight_layout()
plt.savefig('../outputs/plots/feature010_log_transform.png')
plt.show()

## 5. Mutual Information Scores (Feature Importance Prior)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_mi = labeled[feat_cols].fillna(labeled[feat_cols].median())
y_mi = labeled['is_cheating'].astype(int)

mi_scores = mutual_info_classif(X_mi, y_mi, discrete_features=False, random_state=42)
mi_df = pd.Series(mi_scores, index=feat_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
mi_df.plot.bar(ax=ax, color='teal')
ax.set_title('Mutual Information Score vs. is_cheating')
ax.set_ylabel('MI Score')
plt.tight_layout()
plt.savefig('../outputs/plots/mutual_information_scores.png')
plt.show()

print('Top 5 features by MI:')
print(mi_df.head())

## 6. Correlation Heatmap — Feature Redundancy Check

In [ ]:
corr = labeled[feat_cols].fillna(0).corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../outputs/plots/correlation_heatmap.png')
plt.show()

# Flag near-duplicates
high_corr = [(i, j, corr.iloc[i, j])
             for i in range(len(feat_cols))
             for j in range(i+1, len(feat_cols))
             if abs(corr.iloc[i, j]) > 0.85]
if high_corr:
    print('High-correlation pairs (>0.85):')
    for a, b, v in high_corr:
        print(f'  {feat_cols[a]} ↔ {feat_cols[b]}: {v:.3f}')
else:
    print('No near-duplicate feature pairs detected.')

## 7. Social Graph — Degree Distribution

In [ ]:
import networkx as nx

print('Loading social graph (may take ~30s)…')
edges = pd.read_csv(PATHS['graph_file'])
G = nx.from_pandas_edgelist(edges, source='user_a', target='user_b')
print(f'Nodes: {G.number_of_nodes():,}  |  Edges: {G.number_of_edges():,}')

degrees = dict(G.degree())
labeled['graph_degree'] = labeled['user_hash'].map(degrees).fillna(0)

fig, ax = plt.subplots(figsize=(10, 4))
for label, grp in labeled.groupby('is_cheating'):
    np.log1p(grp['graph_degree']).hist(ax=ax, bins=60, alpha=0.6,
                                        label='Cheater' if label==1 else 'Legit',
                                        density=True)
ax.set_title('Degree Distribution: Cheater vs. Legit (log scale)')
ax.set_xlabel('log1p(degree)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/degree_distribution.png')
plt.show()

## 8. Baseline XGBoost — ROC & Cost Curve

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve
from xgboost import XGBClassifier
from src.features.behavioral import build_behavioral_features

lab_feat = build_behavioral_features(labeled)
meta_cols = ['user_hash', 'is_cheating', 'high_conf_clean']
feat_cols_eng = [c for c in lab_feat.columns if c not in meta_cols]

X = lab_feat[feat_cols_eng].astype(float)
y = lab_feat['is_cheating'].astype(int)

xgb_base = XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=5,
    scale_pos_weight=2.0, use_label_encoder=False,
    eval_metric='logloss', n_jobs=-1, random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(xgb_base, X, y, cv=skf, method='predict_proba')[:, 1]

auc = roc_auc_score(y, oof_probs)
print(f'Baseline XGBoost OOF AUC: {auc:.4f}')

t_pass, t_block, best_cost = find_best_thresholds(y.values, oof_probs)
print(f'Optimal t_pass={t_pass:.3f}  t_block={t_block:.3f}  min_cost=${best_cost:,.0f}')

# Save OOF probs for threshold script
oof_df = pd.DataFrame({'user_hash': labeled['user_hash'], 'prediction': oof_probs})
oof_df.to_csv('../outputs/submissions/baseline_oof.csv', index=False)

## 9. Cost vs. Threshold Sensitivity

In [ ]:
# Sweep t_pass with t_block fixed at 0.8
t_pass_range = np.arange(0.01, 0.80, 0.01)
costs = [competition_cost(y.values, oof_probs, t1, 0.8) for t1 in t_pass_range]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_pass_range, costs, linewidth=2)
ax.axvline(t_pass, color='red', linestyle='--', label=f'Optimal t_pass={t_pass:.3f}')
ax.set_xlabel('t_pass threshold')
ax.set_ylabel('Total Cost ($)')
ax.set_title('Cost Sensitivity to t_pass (t_block=0.8 fixed)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/cost_sensitivity.png')
plt.show()